In [ ]:

from google.colab import drive
drive.mount('/content/drive')

import os, gc, warnings
os.environ['CUDA_VISIBLE_DEVICES'] = '-1'  # CPU only — stable
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2
!pip install lime
from lime import lime_image
from skimage.segmentation import mark_boundaries
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

PROJECT_ROOT = '/content/drive/MyDrive/MRI_Brain_Tumor_Project'
CLAHE_DIR    = f'{PROJECT_ROOT}/data/clahe_processed'
IMG_SIZE     = (224, 224)
CLASS_NAMES  = ['glioma', 'meningioma', 'notumor', 'pituitary']
PRED_DIR     = f'{PROJECT_ROOT}/results/predictions'
FIG_DIR      = f'{PROJECT_ROOT}/results/figures'

# Load model
print("Loading model...")
effnet_model = tf.keras.models.load_model(
    f'{PROJECT_ROOT}/models/checkpoints/effnet_correct_s2b.keras'
)
y_true       = np.load(f'{PRED_DIR}/y_true.npy')
y_pred_effnet= np.load(f'{PRED_DIR}/y_pred_effnet_final.npy')
uncertainty  = np.load(f'{PRED_DIR}/uncertainty.npy')

# Load 4 images only
sample_indices = {'glioma':32,'meningioma':570,'notumor':1195,'pituitary':1202}

def load_img(cls_name, img_idx):
    class_order  = sorted(os.listdir(f'{CLAHE_DIR}/Testing'))
    cls_position = class_order.index(cls_name)
    local_idx    = img_idx - cls_position * 400
    files        = sorted(os.listdir(f'{CLAHE_DIR}/Testing/{cls_name}'))
    img = cv2.imread(f'{CLAHE_DIR}/Testing/{cls_name}/{files[local_idx]}')
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img, IMG_SIZE)

data = {}
for cls_name, idx in sample_indices.items():
    img   = load_img(cls_name, idx)
    img_p = tf.keras.applications.efficientnet.preprocess_input(
                img.astype(np.float32))[np.newaxis]
    probs = effnet_model(img_p, training=False).numpy()[0]
    pc    = int(np.argmax(probs))
    data[cls_name] = {
        'img': img, 'img_p': img_p,
        'pred': pc, 'conf': float(probs[pc]),
        'unc':  float(uncertainty[idx])
    }
    print(f"  {cls_name}: pred={CLASS_NAMES[pc]}, conf={probs[pc]:.3f}")

print("\nAll 4 images loaded. Starting XAI...\n")


# 1. GRAD-CAM++

print(" Grad-CAM++ ")
backbone   = effnet_model.layers[1]
last_conv  = [l.name for l in reversed(backbone.layers)
              if isinstance(l, tf.keras.layers.Conv2D)][0]
grad_model = tf.keras.Model(
    inputs=backbone.input,
    outputs=[backbone.get_layer(last_conv).output, backbone.output]
)

def gradcam_pp(img_p, class_idx):
    with tf.GradientTape() as t2:
        with tf.GradientTape() as t1:
            with tf.GradientTape() as t0:
                inp = tf.cast(img_p, tf.float32)
                co, pred = grad_model(inp)
                t0.watch(co); t1.watch(co); t2.watch(co)
                loss = pred[:, class_idx]
            g1 = t0.gradient(loss, co)
        g2 = t1.gradient(g1, co)
    g3 = t2.gradient(g2, co)
    co=co[0]; g1=g1[0]; g2=g2[0]; g3=g3[0]
    s   = tf.reduce_sum(co, axis=(0,1))
    den = 2*g2 + s[None,None,:]*g3
    den = tf.where(den==0, tf.ones_like(den), den)
    w   = tf.reduce_sum((g2/den)*tf.nn.relu(g1), axis=(0,1))
    h   = tf.nn.relu(tf.reduce_sum(w*co, axis=-1)).numpy()
    h   = cv2.resize(h, (224,224))
    if h.max()>0: h=(h-h.min())/(h.max()-h.min())
    return h

import matplotlib
cmap_jet = matplotlib.colormaps['jet']

for cls_name in CLASS_NAMES:
    d  = data[cls_name]
    h  = gradcam_pp(d['img_p'], d['pred'])
    hc = (cmap_jet(h)[:,:,:3]*255).astype(np.uint8)
    ov = cv2.addWeighted(d['img'], 0.55, hc, 0.45, 0)
    data[cls_name]['gradcam'] = ov
    print(f"  {cls_name} done")

# 2. LIME

print("\n LIME ")
explainer = lime_image.LimeImageExplainer(random_state=SEED)

def predict_fn(images):
    p = tf.keras.applications.efficientnet.preprocess_input(
            images.astype(np.float32))
    return effnet_model(p, training=False).numpy()

for cls_name in CLASS_NAMES:
    d   = data[cls_name]
    exp = explainer.explain_instance(
        d['img'].astype(np.uint8), predict_fn,
        top_labels=4, hide_color=0,
        num_samples=300, random_seed=SEED
    )
    temp, mask = exp.get_image_and_mask(
        d['pred'], positive_only=False,
        num_features=6, hide_rest=False
    )
    ov = mark_boundaries(temp.astype(np.uint8), mask.astype(np.int_),
                         color=(1,0,0), outline_color=(0,0,0))
    data[cls_name]['lime'] = (ov*255).astype(np.uint8)
    print(f"  {cls_name} done")

# 3. INTEGRATED GRADIENTS

print("\n Integrated Gradients ")

def integrated_grads(model, img_p, class_idx, n_steps=20):
    baseline = np.zeros_like(img_p)
    attrs    = np.zeros((224,224,3), dtype=np.float64)
    for alpha in np.linspace(0.0, 1.0, n_steps):
        interp = tf.constant(baseline + alpha*(img_p-baseline), dtype=tf.float32)
        with tf.GradientTape() as tape:
            tape.watch(interp)
            score = model(interp, training=False)[0, class_idx]
        attrs += tape.gradient(score, interp).numpy()[0]
    attrs  = (img_p[0]-baseline[0]) * attrs / n_steps
    attr2d = np.sum(np.abs(attrs), axis=-1)
    if attr2d.max()>0:
        attr2d = (attr2d-attr2d.min())/(attr2d.max()-attr2d.min())
    return attr2d

for cls_name in CLASS_NAMES:
    d    = data[cls_name]
    attr = integrated_grads(effnet_model, d['img_p'], d['pred'])
    data[cls_name]['ig'] = attr
    print(f"  {cls_name} done")

#
# 4. FINAL 4- FIGURE

print("\nBuilding 4-panel figure")

fig, axes = plt.subplots(4, 4, figsize=(18, 18))
row_titles = ['Original MRI', 'Grad-CAM++', 'LIME', 'Integrated Gradients']

for col, cls_name in enumerate(CLASS_NAMES):
    d = data[cls_name]

    # Row 0: Original
    axes[0,col].imshow(d['img'])
    axes[0,col].set_title(
        f'{cls_name.capitalize()}\n'
        f'Pred: {CLASS_NAMES[d["pred"]]} ({d["conf"]:.1%})\n'
        f'Unc: {d["unc"]:.4f}', fontsize=9)
    axes[0,col].axis('off')

    # Row 1: Grad-CAM++
    axes[1,col].imshow(d['gradcam'])
    axes[1,col].set_title(f'Grad-CAM++', fontsize=9)
    axes[1,col].axis('off')

    # Row 2: LIME
    axes[2,col].imshow(d['lime'])
    axes[2,col].set_title(f'LIME', fontsize=9)
    axes[2,col].axis('off')

    # Row 3: Integrated Gradients
    axes[3,col].imshow(d['img'])
    axes[3,col].imshow(d['ig'], cmap='hot', alpha=0.5, vmin=0, vmax=1)
    axes[3,col].set_title(f'Integrated Gradients', fontsize=9)
    axes[3,col].axis('off')

for row, title in enumerate(row_titles):
    axes[row,0].set_ylabel(title, fontsize=12, fontweight='bold', labelpad=10)

plt.suptitle('XAI Panel — EfficientNetB3 Brain Tumor Classification',
             fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/xai_4panel_final.png', dpi=150, bbox_inches='tight')
plt.show()


print(f"4-panel figure saved to: {FIG_DIR}/xai_4panel_final.png")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading model...
  glioma: pred=glioma, conf=0.997
  meningioma: pred=meningioma, conf=0.922
  notumor: pred=notumor, conf=0.998
  pituitary: pred=pituitary, conf=1.000

All 4 images loaded. Starting XAI...

=== Grad-CAM++ ===
  glioma done
  meningioma done
  notumor done
  pituitary done

=== LIME ===


  0%|          | 0/300 [00:00<?, ?it/s]

  glioma done


  0%|          | 0/300 [00:00<?, ?it/s]

KeyboardInterrupt: 